In [70]:
import numpy as np
import pandas as pd

In [72]:
df_yonalishlar = pd.read_excel('../data/raw/Fanlar_majmuasi_2025-2026.xlsx')
df_abts = pd.read_csv('../data/raw/abt_unique.csv')

In [75]:
df_yonalishlar.head()

,Yoʻnalish kodi,Bakalavriat taʼlim yoʻnalishlari nomi,1-fan,2-fan
0,60110100,Pedagogika,Tarix,Ona tili va adabiyoti
1,60110200,Maktabgacha taʼlim,Biologiya,Ona tili va adabiyoti
2,60110300,Maxsus pedagogika: logopediya,Biologiya,Ona tili va adabiyoti
3,60110300,Maxsus pedagogika: olegofrenopedagogika,Biologiya,Ona tili va adabiyoti
4,60110300,Maxsus pedagogika: surdopedagogika,Biologiya,Ona tili va adabiyoti


In [78]:
df_yonalishlar.info()

<class 'pandas.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 4 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Yoʻnalish kodi                         218 non-null    int64
 1   Bakalavriat taʼlim yoʻnalishlari nomi  218 non-null    str  
 2   1-fan                                  218 non-null    str  
 3   2-fan                                  218 non-null    str  
dtypes: int64(1), str(3)
memory usage: 6.9 KB


In [81]:
df_abts.head()

,ismi,ID,otm_nomi,yonalishi,shifr_kodi,talim_tili,talim_shakli,toplagan_bali
0,URAKBAYEV N. M.,6042695,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0
1,NAZEROV I. A.,6045467,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0
2,TENGELOV X. I.,6872807,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0
3,JANABAYEV S. O.,6259423,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,186.7
4,ERDOSHEVA X. T.,6980903,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,185.7


In [82]:
df_abts.info()

<class 'pandas.DataFrame'>
RangeIndex: 357561 entries, 0 to 357560
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   ismi           357561 non-null  str    
 1   ID             357561 non-null  int64  
 2   otm_nomi       357561 non-null  str    
 3   yonalishi      357561 non-null  str    
 4   shifr_kodi     357561 non-null  int64  
 5   talim_tili     357561 non-null  str    
 6   talim_shakli   357559 non-null  str    
 7   toplagan_bali  357561 non-null  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 21.8 MB


In [92]:
royxat1 = df_abts['yonalishi'].unique()
royxat2 = df_yonalishlar['Bakalavriat taʼlim yoʻnalishlari nomi'].unique()

# Har ehtimolga qarshi hammasini kichik harfga o'tkazib to'plamga aylantiramiz
set1 = set([str(x).lower().strip() for x in royxat1])
set2 = set([str(x).lower().strip() for x in royxat2])

print("Ikkalasida ham borlari (Kesishmasi):", set1.intersection(set2))
print("Faqat 1-ro'yxatda bor, lekin 2-da yo'qlar:", set1.difference(set2))
print("Faqat 2-ro'yxatda bor, lekin 1-da yo'qlar:", set2.difference(set1))


Ikkalasida ham borlari (Kesishmasi): {'qurilish muhandisligi', 'biologiya', 'astronomiya', 'yer kadastri va yer tuzish', 'oʻzbek tili va adabiyoti', 'ona tili va adabiyoti: qozoq tili', 'xorijiy til va adabiyoti: nemis tili', 'bank ishi', 'veterinariya sanitariya ekspertizasi', 'kompyuter lingvistikasi', 'muzeyshunoslik: muzey menejmenti va madaniy turizm', 'parfyumeriya-kosmetika mahsulotlari texnologiyasi', 'chaqiriqqacha harbiy taʼlim', 'tarjima nazariyasi va amaliyoti: xitoy tili', 'logistika', 'filologiya va tillarni oʻqitish: nemis tili', 'hayot faoliyati xavfsizligi', 'kiberxavfsizlik injiniringi', 'jurnalistika', 'axborot tizimlari va texnologiyalari', 'muzeyshunoslik: muzey predmetlari ekspertizasi, konservatsiyalash va taʼmirlash', 'filologiya va tillarni oʻqitish: qozoq tili', 'tarjima nazariyasi va amaliyoti: nemis tili', 'maxsus pedagogika: tiflopedagogika', 'suv taʼminoti muhandislik tizimlari', 'tasviriy sanʼat va muhandislik grafikasi', 'filologiya va tillarni oʻqitish:

In [93]:
# 1. df_abts dagi 'yonalishi' ustunini vaqtinchalik tozalab, 'topilmaganlar' ro'yxatida mavjudligini tekshiramiz
xato_qatorlar_maskasi = df_abts['yonalishi'].astype(str).str.strip().str.lower().isin(set1.difference(set2))

# 2. ~ (tilde) belgisi yordamida ushbu XATO qatorlarni olib tashlab, faqat to'g'rilarini df_abts ga qayta saqlaymiz
df_abts = df_abts[~xato_qatorlar_maskasi]

# Natijani tekshirish
print("Xato qatorlar olib tashlangandan keyingi jami qatorlar soni:", len(df_abts))


Xato qatorlar olib tashlangandan keyingi jami qatorlar soni: 344701


In [94]:
# 1. Ikkala jadvalda ham barcha harflari kichik va probellari tozalangan yordamchi (join_key) ustun yaratamiz
df_abts['join_key'] = df_abts['yonalishi'].astype(str).str.strip().str.lower()
df_yonalishlar['join_key'] = df_yonalishlar['Bakalavriat taʼlim yoʻnalishlari nomi'].astype(str).str.strip().str.lower()

# 2. Shu tozalangan yordamchi ustun bo'yicha LEFT JOIN qilamiz
merged_df = pd.merge(
    df_abts,
    df_yonalishlar,
    how='left',
    on='join_key'  # Birlashtirish shu ustun asosida bo'ladi
)

# 3. Birlashtirish muvaffaqiyatli yakunlangach, jadvaldagi yordamchi 'join_key' ustunini o'chirib tashlaymiz
merged_df = merged_df.drop(columns=['join_key'])

# (Ixtiyoriy) Asl jadvallar ham toza turishi uchun ulardan ham o'chirib qo'yamiz
df_abts = df_abts.drop(columns=['join_key'])
df_yonalishlar = df_yonalishlar.drop(columns=['join_key'])

# Natijani tekshirish uchun dastlabki 5 ta qatorni ko'ramiz
merged_df.head()


,ismi,ID,otm_nomi,yonalishi,shifr_kodi,talim_tili,talim_shakli,toplagan_bali,Yoʻnalish kodi,Bakalavriat taʼlim yoʻnalishlari nomi,1-fan,2-fan
0,URAKBAYEV N. M.,6042695,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0,60540200,Amaliy matematika,Matematika,Fizika
1,NAZEROV I. A.,6045467,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0,60540200,Amaliy matematika,Matematika,Fizika
2,TENGELOV X. I.,6872807,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,189.0,60540200,Amaliy matematika,Matematika,Fizika
3,JANABAYEV S. O.,6259423,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,186.7,60540200,Amaliy matematika,Matematika,Fizika
4,ERDOSHEVA X. T.,6980903,Nukus davlat pedagogika instituti,Amaliy matematika,60540200,O'zbekcha,Kunduzgi,185.7,60540200,Amaliy matematika,Matematika,Fizika


In [95]:
merged_df.shape

(344701, 12)

In [96]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 344701 entries, 0 to 344700
Data columns (total 12 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   ismi                                   344701 non-null  str    
 1   ID                                     344701 non-null  int64  
 2   otm_nomi                               344701 non-null  str    
 3   yonalishi                              344701 non-null  str    
 4   shifr_kodi                             344701 non-null  int64  
 5   talim_tili                             344701 non-null  str    
 6   talim_shakli                           344701 non-null  str    
 7   toplagan_bali                          344701 non-null  float64
 8   Yoʻnalish kodi                         344701 non-null  int64  
 9   Bakalavriat taʼlim yoʻnalishlari nomi  344701 non-null  str    
 10  1-fan                                  344701 non-null  str    
 11

In [97]:
merged_df = merged_df[['ismi', 'ID', 'toplagan_bali', '1-fan', '2-fan']]

In [98]:
merged_df.head()

,ismi,ID,toplagan_bali,1-fan,2-fan
0,URAKBAYEV N. M.,6042695,189.0,Matematika,Fizika
1,NAZEROV I. A.,6045467,189.0,Matematika,Fizika
2,TENGELOV X. I.,6872807,189.0,Matematika,Fizika
3,JANABAYEV S. O.,6259423,186.7,Matematika,Fizika
4,ERDOSHEVA X. T.,6980903,185.7,Matematika,Fizika


In [99]:
merged_df.to_csv('../data/processed/real_abts.csv', index=False)